# Manual Day — Optimization & the Cosmological Constant

Closed-agent, ~60 min. Complete every `# FILL IN` cell, then run the check.

**Four problems. Work in order, as many as time permits.** The last one is a challenge.

## Mini-lecture: optimization without gradients

Gradient descent has carried us this far, but it needs a derivative and it only walks
downhill. Today's objective is built out of integers, so there's no derivative to take,
and it has an enormous number of local minima to get stuck in. Two techniques that don't
care: simulated annealing and genetic algorithms.

### The cosmological constant

The observed cosmological constant is about $10^{-120}$ in Planck units. Nothing in
effective field theory explains a number like that. Every field you add kicks in vacuum
energy of order its cutoff, and those contributions have no reason to cancel to 120
digits.

Bousso and Polchinski proposed in 2000 that they don't cancel. They just come in enough
varieties that some combination lands near zero by accident. In their model

$$\Lambda = \Lambda_0 + N\cdot N, \qquad N \in \mathbb{Z}^{h_{21}}, \qquad \Lambda_0 < 0,$$

where $\Lambda_0$ is a large negative constant and $N$ is a vector of flux quanta,
integers because fluxes are quantized. Every choice of $N$ gives you a vacuum of the
theory. Once $h_{21}$ is large there are exponentially many flux vectors to choose from,
and the $\Lambda$ values they produce are packed densely enough to make up not a
continuum but a *discretuum*, fine enough that some vacua land in the thin window
$0 < \Lambda < \epsilon$ that we observe.

That's the proposal. The small number doesn't come out of a calculation, it gets picked
out of an enormous list of possibilities. But a list that contains what you want is no
use if you can't find the entry, and here finding it means finding an integer vector
whose norm-squared sits inside a shell of thickness $\epsilon$ around a sphere in 1000
dimensions. Problem 1 is random guessing and Problem 2 is annealing, and six orders of magnitude
separate them.

### Simulated annealing

Treat your objective as an energy $E$ and run the system at a temperature $T$. What
follows is the Metropolis rule with $T$ sliding toward zero.

1. Pick a random $N$. Set $T = T_0$, and store a copy as $N_\text{best}$.
2. Pick a random coordinate $i$ and nudge $N_i$ by a small random integer, clipped back
   into the box. Call the result $N'$.
3. Compute $\Delta E = E(N') - E(N)$.
4. If $\Delta E < 0$, accept $N'$. Otherwise accept it anyway with probability
   $e^{-\Delta E / T}$.
5. If $E(N') < E(N_\text{best})$, update $N_\text{best}$. Keep this separate from step 4.
   The walk is free to wander away from the best point it found and will not bring you
   back to it, so you have to save it yourself.
6. Cool: $T_{k+1} = \rho\, T_k$, with $\rho$ a little under 1.
7. Go back to step 2.

The cooling schedule is the knob you'll spend your time on. While $T$ is high almost
every proposal gets accepted and the walk goes wherever it likes; by the time $T$ is low
only downhill moves survive and the state settles. Cool too fast and you freeze into
whatever basin you started near, which is greedy descent with extra steps. Cool too
slowly and the iterations run out while you're still wandering.

Nothing in those seven steps used a derivative. All they needed was a way to propose a
nearby state, here step 2, and an energy you can evaluate. That's why this works on a
problem made of integers.

### Genetic algorithms

A different idea: carry a whole population instead of a single state, and build new
candidates by splicing old ones together.

1. Start with a population of $P$ random vectors.
2. Score every member by a fitness, say $1/E$, so low energy scores well.
3. Take the two fittest members as parents.
4. Cross over: pick a random cut point $c$, and build the child from the first $c$
   coordinates of one parent and the remaining coordinates of the other.
5. Mutate: with some small probability, replace one random coordinate of the child with
   a random value.
6. Drop the least fit member of the population and put the child in its place.
7. Go back to step 2.

Step 4 has no counterpart in annealing, and it's the reason a genetic algorithm has a
shot at this problem. Our energy is $|\sum_i N_i^2 - \Lambda_0|$, a plain sum over
coordinates, so a parent that has its first 400 coordinates working hands that partial
sum to its child intact. Splicing two parents really does combine their work. When the
objective doesn't break apart that way, crossover mostly just scrambles, and genetic
algorithms are a lot less impressive than the biology story makes them sound. Worth
working out which case you're in before you reach for one.

In [ ]:
import numpy as np

np.random.seed(7)  # reproducible -- run the cells top-to-bottom

# Bousso-Polchinski toy model: choose an integer vector N so that ||N||^2 approaches L0.
L0    = 200000.1293123   # target value of ||N||^2
dim   = 1000             # number of integer coordinates (h_21)
bound = 16               # each coordinate is restricted to [-bound, bound]

# Objective ("energy"): how far ||N||^2 sits from L0.  Smaller is better.
def E(N):
    return abs(np.sum(N**2) - L0)

# Simulated-annealing hyperparameters.
initial_temp = 1000
cooling_rate = 0.99
n_iter       = 30000     # modest -- runs in about a second

## Problem 1 — Random search, and why it can't work

Before anything clever, try the obvious thing. Draw `num_samples` integer vectors
uniformly from the box, score each one with `E`, and keep whichever lands closest to
`L0`. Write `random_search` below.

**Known answer.** This fails, and you can say how badly before you run it. Each
coordinate is drawn uniformly from $\{-16, \dots, 16\}$, so

$$\langle N_i^2 \rangle = \frac{1}{33}\sum_{k=-16}^{16} k^2 = \frac{272}{3} = 90.667,$$

which puts $\langle \|N\|^2 \rangle = 1000 \times 90.667 = 90667$, with a standard
deviation of $2561$. The target sits $42.7$ standard deviations above that. You should land near $10^5$,
about 50% error. Sampling harder doesn't rescue it: I ran $10^4$, $10^5$ and $10^6$
draws and got 99913, 98490 and 96936, so a hundredfold more work bought 3%.

In [ ]:
def random_search(num_samples=10000):
    """Draw num_samples random integer vectors; return the best one and its distance."""
    best_N, best_dist = None, float('inf')
    # FILL IN: loop num_samples times, draw N = np.random.randint(-bound, bound + 1, dim),
    # score it with E(N), and keep whichever vector has the smallest distance so far.
    pass  # <-- replace this with your implementation
    return best_N, best_dist


best_N, baseline_dist = random_search()
print("random search -> distance:", baseline_dist)
print("random search -> percent error:", 100 * baseline_dist / L0)

In [ ]:
# CHECK -- run after filling in the cell above.
k   = np.arange(-bound, bound + 1)
mu  = dim * (k**2).mean()
sig = np.sqrt(dim * ((k**4).mean() - (k**2).mean()**2))
print(f"random search -> distance {baseline_dist:.1f}   (percent error {100*baseline_dist/L0:.1f}%)")
print()
print(f"<N_i^2> = 272/3 = {(k**2).mean():.4f}, so a random vector has ||N||^2 near")
print(f"{mu:.0f} give or take {sig:.0f}, while the target is L0 = {L0:.0f}.")
print(f"That is {(L0-mu)/sig:.1f} standard deviations away.")
print()
print("The trouble isn't that random search is slow. It samples from a distribution")
print("whose mean is less than half of L0, so more draws only buy you its upper tail.")
print("Problems 2 and 3 search instead of sampling.")
assert np.isfinite(baseline_dist), "random_search returned no vector: is the loop filled in?"
assert best_N.shape == (dim,) and np.abs(best_N).max() <= bound, "N must be dim integers inside the box"
assert E(best_N) == baseline_dist, "the vector you return should be the one you scored"
assert 5e4 < baseline_dist < 1.5e5, "a 10^4-sample random search should land near 10^5"

## Problem 2 — Simulated annealing on the Bousso-Polchinski integer packing

Write `simulated_annealing` below, with `E(N) = |sum(N**2) - L0|` as the energy. On each
step, copy the current vector, nudge **one** random coordinate by a small random integer,
and clip it back into `[-bound, bound]`. Then accept or reject, cool, and keep track of
the best vector you ever saw, exactly as in the lecture above.

**Known answer.** `||N||^2` is always an integer, so the closest it can get to
`L0 = 200000.1293123` is `200000`, a distance of `0.1293` (percent error ~6.5e-5%).
Annealing should land on that floor, far below the random-search baseline.

In [ ]:
def simulated_annealing():
    # FILL IN: the simulated-annealing loop (perturb one coord, accept/reject, track best, cool).
    #
    # --- given scaffold: initialize the current and best-ever states ---
    N_cur = np.random.randint(-bound, bound + 1, dim)
    E_cur = E(N_cur)
    N_best, E_best = np.copy(N_cur), E_cur
    T = initial_temp

    for _ in range(n_iter):
        # FILL IN one simulated-annealing step:
        #   1. propose N_new: copy N_cur, add np.random.randint(-2, 3) to ONE random coord,
        #      then np.clip into [-bound, bound]; compute E_new = E(N_new)
        #   2. if E_new < E_best: update N_best, E_best      (best-ever tracking)
        #   3. accept if E_new < E_cur or np.random.rand() < np.exp((E_cur - E_new) / T)
        #   4. cool: T *= cooling_rate
        pass  # <-- replace this with your implementation

    return N_best, E_best


N_best, sa_dist = simulated_annealing()
print("simulated annealing -> distance:", sa_dist)
print("simulated annealing -> percent error:", 100 * sa_dist / L0)

In [ ]:
# CHECK -- run after filling in the cell above.
print(f"random search        distance = {baseline_dist:.4f}  (percent error {100*baseline_dist/L0:.4g}%)")
print(f"simulated annealing  distance = {sa_dist:.4f}  (percent error {100*sa_dist/L0:.4g}%)")
print()
print("Integer floor: ||N||^2 is an integer, so the best possible is |200000 - L0| = 0.1293,")
print("i.e. percent error ~6.5e-5%. Simulated annealing should sit at/near this floor")
print(f"and beat the random-search baseline by a wide margin (here ~{baseline_dist / sa_dist:.0f}x smaller).")
assert sa_dist < 1.0, "annealing should reach the integer floor at 0.1293"
assert sa_dist < baseline_dist / 1000, "and leave the random-search baseline far behind"

## Problem 3 — A genetic algorithm on the same packing

Same target and the same floor, now with a different algorithm. Write
`genetic_algorithm` below, following
the steps from the lecture: score the population, take the two fittest as parents, cut at
a random point and splice, mutate a coordinate with probability `mutation_rate`, and
overwrite the least fit member with the child.

Fitness is $1/E$, so ranking by fitness and ranking by energy give the same order,
reversed. Work with the energies directly if you prefer.

**Known answer.** The floor is the `0.1293` from Problem 2, because the target hasn't
moved. The genetic algorithm gets within a unit or two of it and then stops improving,
usually with most of the generations still to go. The plot and the check below are there
so you can see where it stalls and measure why.

In [ ]:
# GIVEN setup for Problem 3 -- run this before the cell below.
# Same E, L0, dim and bound as Problems 1 and 2.  Only the algorithm changes.
np.random.seed(1)  # reproducible, independent of the runs above

pop_size      = 100     # members of the population
generations   = 50000   # one child per generation -- runs in about half a second
mutation_rate = 0.8     # chance of mutating one coordinate of each child

In [ ]:
def genetic_algorithm():
    # FILL IN: the genetic-algorithm loop (select, cross over, mutate, replace worst).
    #
    # --- given scaffold: a random starting population and its energies ---
    pop     = np.random.randint(-bound, bound + 1, (pop_size, dim))
    E_pop   = np.array([E(ind) for ind in pop])   # E_pop[k] is the energy of member k
    history = [E_pop.min()]

    for _ in range(generations):
        # FILL IN one generation:
        #   1. parents: the two members with the smallest E_pop
        #      (np.argsort(E_pop) ranks them; fitness 1/E gives the same order)
        #   2. crossover: c = np.random.randint(1, dim), then
        #      child = np.concatenate([parent1[:c], parent2[c:]])
        #   3. mutate: with probability mutation_rate, set ONE random coordinate of the
        #      child to np.random.randint(-bound, bound + 1)
        #   4. replace: overwrite the member with the LARGEST E_pop with the child,
        #      and set that entry of E_pop to E(child)
        #   5. history.append(E_pop.min())
        pass  # <-- replace this with your implementation

    best = np.argmin(E_pop)
    return pop[best], E_pop[best], history, pop


N_ga, ga_dist, history, final_pop = genetic_algorithm()
print("genetic algorithm -> distance:", ga_dist)
print("genetic algorithm -> percent error:", 100 * ga_dist / L0)

In [ ]:
# GIVEN plot -- best-ever distance per generation, against the integer floor.
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.semilogy(history, label="genetic algorithm")
plt.axhline(abs(200000 - L0), ls="--", c="k", label="integer floor, 0.1293")
plt.xlabel("generation")
plt.ylabel("best-ever distance  (log scale)")
plt.title("Bousso-Polchinski: genetic-algorithm convergence")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CHECK -- run after filling in the cell above.
floor = abs(200000 - L0)
print(f"random search        distance = {baseline_dist:.4f}")
print(f"simulated annealing  distance = {sa_dist:.4f}")
print(f"genetic algorithm    distance = {ga_dist:.4f}      (integer floor = {floor:.4f})")
print()

plateau = int(np.argmin(history))
pairs = [np.mean(final_pop[a] != final_pop[b])
         for a in range(0, pop_size, 7) for b in range(a + 1, pop_size, 7)]
print(f"The best-ever value stopped improving at generation {plateau} of {generations}.")
print(f"By the end, two members of the population differ on {100*np.mean(pairs):.2f}% of")
print("their 1000 coordinates: the population has collapsed onto a single vector. With")
print("nothing left to recombine, crossover does nothing and the only move still")
print("available is one random mutation, which almost never helps. That is the plateau")
print("you see in the plot.")
print()
print("Annealing lands on the floor because its proposal is a small nudge to one")
print("coordinate, so it can fine-tune. The mutation here replaces a coordinate outright.")
assert ga_dist < 5, "the genetic algorithm should land within a few units of the floor"
assert ga_dist < baseline_dist / 1000, "it should beat random search by a wide margin"

## Problem 4 — Challenge: charges on a sphere

J. J. Thomson asked in 1904 where $n$ electrons sit on a sphere if they repel each other
and can't leave the surface. Minimize

$$U = \sum_{i<j} \frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}, \qquad |\mathbf{r}_i| = 1.$$

Nothing is quantized here, so unlike everything so far this is a continuous problem, and
the new ingredient is the constraint. Anneal it the same way as before, with one extra
move:
perturb one charge by a small Gaussian, then divide it by its norm to put it back on the
sphere before you score it.

One wrinkle that matters here: scale the proposal with the temperature, using
`step * np.sqrt(T)`, so the moves shrink as the system cools. With a fixed step the walk
freezes early, since once $T$ is tiny a jump of size 0.3 is rejected essentially every
time. Switching to a shrinking step improves the $n = 12$ energy by a factor of about 40.

**Known answers.** For $n = 4$ the charges go to the corners of a regular tetrahedron,
giving $U = 3\sqrt{6}/2 = 3.674235$. For $n = 12$ they go to an icosahedron, with
$U = 49.165253$. Both are exact to the digits shown, and you should land within $10^{-3}$
of the first and $10^{-2}$ of the second. The $n = 4$ case has a second check that needs
no reference value at all: all six pairwise distances should come out equal, to
$\sqrt{8/3} = 1.632993$.

In [ ]:
def U(P):
    """Coulomb energy of unit charges sitting at the rows of P."""
    # FILL IN: the pairwise distances, then the sum of 1/r over distinct pairs.
    #   np.linalg.norm(P[:, None, :] - P[None, :, :], axis=-1) gives the full matrix,
    #   and np.triu_indices(len(P), 1) picks out each pair once.
    pass  # <-- replace this with your implementation

def thomson(n, iters=40000, T0=1.0, cooling=0.9997, step=0.3):
    np.random.seed(3)
    P = np.random.randn(n, 3)
    P /= np.linalg.norm(P, axis=1, keepdims=True)   # start somewhere on the sphere
    U_cur, T = U(P), T0

    for _ in range(iters):
        # FILL IN one annealing step:
        #   1. copy P, move ONE random charge by step * np.sqrt(T) * np.random.randn(3)
        #   2. divide that charge by its norm so it sits back on the sphere
        #   3. accept by the same Metropolis rule as Problem 2, using U
        #   4. cool: T *= cooling
        pass  # <-- replace this with your implementation
    return P, U(P)


P4,  U4  = thomson(4)
P12, U12 = thomson(12)
print("n =  4 ->", U4)
print("n = 12 ->", U12)

In [ ]:
# CHECK -- run after filling in the cell above.
assert U4 is not None and U12 is not None, "U(P) has to return the energy, not None"
exact4, exact12 = 3 * np.sqrt(6) / 2, 49.16525306
print(f"n =  4:  U = {U4:.6f}   exact 3*sqrt(6)/2    = {exact4:.6f}   error {abs(U4-exact4):.1e}")
print(f"n = 12:  U = {U12:.6f}   exact icosahedron = {exact12:.6f}   error {abs(U12-exact12):.1e}")
print()
d6 = np.linalg.norm(P4[:, None, :] - P4[None, :, :], axis=-1)[np.triu_indices(4, 1)]
print("the six pairwise distances at n = 4:", np.round(d6, 4))
print(f"a regular tetrahedron on the unit sphere has all six equal to sqrt(8/3) = {np.sqrt(8/3):.6f},")
print("so this one you can confirm without looking up a number.")
assert abs(U4 - exact4) < 1e-3, "n = 4 should reach the tetrahedron energy"
assert abs(U12 - exact12) < 1e-2, "n = 12 should reach the icosahedron energy"
assert np.ptp(d6) < 2e-2, "the four charges should be equidistant"